In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import torch
import numpy as np
from transformers import BertModel, BertTokenizer, AutoTokenizer,DPRContextEncoder, DPRQuestionEncoder
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Motivation on sentence embedding

Below we load in token embedding from Bert Uncased model and use mean pooling to see how bad the performance is without sentence embedding training leveraing contrastive loss 

In [6]:
bert_model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
bert_model = BertModel.from_pretrained(bert_model_name)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [7]:
def get_sembedding_mean_pooling(sentence):
    encoded_input = tokenizer(sentence, padding=True, truncation=True, return_tensors="pt")
    attention_mask = encoded_input["attention_mask"]

    with torch.no_grad():
        output = bert_model(**encoded_input)
    
    token_embeddings = output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float() 

    sentence_embedding = torch.sum(token_embeddings * input_mask_expanded, 1)/torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    return sentence_embedding.flatten().tolist()



In [8]:
messages = [
    "My phone needs upgrade.",
    "My favorite activity is hiking.",
    "music calms my soul.",
    "How is the weather like tomorrow?",
    "The future is bright. What do you think?"
]

In [9]:
embeddings = []
for msg in messages:
    embeddings.append(get_sembedding_mean_pooling(msg))


In [10]:
for i in range(len(messages)):
    for j in range(i+1, len(messages)):
        print(f"The similarity between messages '{messages[i]}' and "
              f"'{messages[j]}' is {cosine_similarity([embeddings[i]],
                                    [embeddings[j]])[0][0]}\n")

The similarity between messages 'My phone needs upgrade.' and 'My favorite activity is hiking.' is 0.6319901593144563

The similarity between messages 'My phone needs upgrade.' and 'music calms my soul.' is 0.6891224515536771

The similarity between messages 'My phone needs upgrade.' and 'How is the weather like tomorrow?' is 0.5522841322438776

The similarity between messages 'My phone needs upgrade.' and 'The future is bright. What do you think?' is 0.5436160732438446

The similarity between messages 'My favorite activity is hiking.' and 'music calms my soul.' is 0.7055779837506785

The similarity between messages 'My favorite activity is hiking.' and 'How is the weather like tomorrow?' is 0.5586192011261311

The similarity between messages 'My favorite activity is hiking.' and 'The future is bright. What do you think?' is 0.6210847394143679

The similarity between messages 'music calms my soul.' and 'How is the weather like tomorrow?' is 0.6102433955102697

The similarity between me

### We need SBERT and Dual Encoders 

Dual encoders uses contrastive loss to train the SBERT where it tries to encourage the similar sentences to be closer while different sentences to be further apart

model link: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [7]:
dual_encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
sbert_model = SentenceTransformer(dual_encoder_name)


In [12]:
s_embeddings = []
for msg in messages:
    s_embeddings.append(list(sbert_model.encode(msg)))

In [13]:
for i in range(len(messages)):
    for j in range(i+1, len(messages)):
        print(f"The similarity between messages '{messages[i]}' and "
              f"'{messages[j]}' is {cosine_similarity([s_embeddings[i]],
                                    [s_embeddings[j]])[0][0]}\n")

The similarity between messages 'My phone needs upgrade.' and 'My favorite activity is hiking.' is 0.0903390571475029

The similarity between messages 'My phone needs upgrade.' and 'music calms my soul.' is 0.18195918202400208

The similarity between messages 'My phone needs upgrade.' and 'How is the weather like tomorrow?' is 0.10489782691001892

The similarity between messages 'My phone needs upgrade.' and 'The future is bright. What do you think?' is 0.11169230937957764

The similarity between messages 'My favorite activity is hiking.' and 'music calms my soul.' is 0.22299325466156006

The similarity between messages 'My favorite activity is hiking.' and 'How is the weather like tomorrow?' is 0.17192156612873077

The similarity between messages 'My favorite activity is hiking.' and 'The future is bright. What do you think?' is 0.058339621871709824

The similarity between messages 'music calms my soul.' and 'How is the weather like tomorrow?' is 0.133857861161232

The similarity betw

### Train dual encoder with constrative loss

$$ \mathcal{L} = (1 - Y)\frac{1}{2} (D_w)^2 + (Y)\frac{1}{2} {\max(0, m - D_w) }^2 $$

Where:

$D_w$: The Euclidean distance between the two embeddings.

$Y$: The label ($0$ for similar, $1$ for dissimilar - Note: libraries vary on which label is 0 or 1).

$m$: The margin. If dissimilar pairs are already pushed past this distance, the loss is zero (we stop worrying about them).


we can use cross entropy to get this loss in training

$$ \mathcal{L} =- \frac{1}{N}\sum_{i=1}^{N}\log \left(\frac{e^{\text{sim}(a_i, p_i)/ \tau}}{\sum_{j=1}^{N} e^{\text{sim}(a_i, p_j)/ \tau}}\right) $$

Where:

$N$: The batch size.

$a_i$: The anchor sentence embedding for the $i$-th example.

$p_i$: The positive sentence embedding for the $i$-th example.

$p_j$: The positive candidate from the $j$-th pair. When $i \neq j$, $p_j$ serves as a negative example for $a_i$.

$\text{sim}(u, v)$: The similarity function (usually Cosine Similarity: $\frac{u \cdot v}{||u|| ||v||}$).

$\tau$ (Tau): A temperature hyperparameter (scales the logits).



In [14]:
df = pd.DataFrame(
    [
        [4.3, 1.2, 0.05, 1.07],
        [0.18, 3.2, 0.09, 0.05],
        [0.85, 0.27, 2.2, 1.03],
        [0.23, 0.57, 0.12, 5.1]
    ]
)
data = torch.tensor(df.values, dtype=torch.float32)

In [15]:
df

,0,1,2,3
0,4.30,1.20,0.05,1.07
1,0.18,3.20,0.09,0.05
2,0.85,0.27,2.20,1.03
3,0.23,0.57,0.12,5.10


In [24]:
data.shape, target.shape

(torch.Size([4, 4]), torch.Size([4]))

In [17]:
def contrastive_loss(data):
    target = torch.arange(data.size(0))
    loss = torch.nn.CrossEntropyLoss()(data, target)
    return loss

In [22]:
target = torch.arange(data.size(0))

In [23]:
torch.nn.CrossEntropyLoss()(data, target)

tensor(0.1966)

Based on your description, you have a Batch Size ($N$) of 4 and Number of Classes ($C$) of 4.

Input ($x$): A $4 \times 4$ matrix of raw logits (unnormalized scores).
Target ($y$): A vector of size 4 containing the correct class indices (integers from 0 to 3).
In PyTorch, nn.CrossEntropyLoss combines LogSoftmax and NLLLoss (Negative Log Likelihood) in one single class.

The Formula
The loss is calculated for each example in the batch and then averaged (by default).

For a specific row $i$ (where $i$ goes from 0 to 3) and its corresponding correct class label $y_i$:

$$ \text{loss}(x_i, y_i) = -\log\left( \frac{\exp(x_{i, y_i})}{\sum_{j=0}^{3} \exp(x_{i, j})} \right) $$

This can be simplified algebraically to:

$$ \text{loss}(x_i, y_i) = -x_{i, y_i} + \log\left( \sum_{j=0}^{3} \exp(x_{i, j}) \right) $$

The final total loss is the average over the batch:

$$ \text{Total Loss} = \frac{1}{4} \sum_{i=0}^{3} \text{loss}(x_i, y_i) $$

Step-by-Step Breakdown
Exponentiate: Calculate $e^{x}$ for every element in the $4 \times 4$ matrix.
Sum: For each row, sum these exponential values (the denominator of Softmax).
Log: Take the logarithm of that sum.
Subtract: Subtract the raw logit of the correct class ($x_{i, y_i}$) from that log-sum.
Average: Sum these results for all 4 rows and divide by 4.

Encoder Module

In [25]:
class Encoder(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, output_embed_dim):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocab_size, embed_dim)
        self.encoder = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(embed_dim, nhead=8, batch_first=True),
            num_layers=3,
            norm=torch.nn.LayerNorm([embed_dim]),
            enable_nested_tensor=False
        )
        self.projection = torch.nn.Linear(embed_dim, output_embed_dim)
    
    def forward(self, tokenizer_output):
        x = self.embedding_layer(tokenizer_output['input_ids'])
        x = self.encoder(x, src_key_padding_mask=tokenizer_output['attention_mask'].logical_not())
        cls_embed = x[:,0,:]
        return self.projection(cls_embed)

In [26]:
def train_loop(dataset):
    embed_size = 512
    output_embed_size = 128
    max_seq_len = 64
    batch_size = 32

    # define the question/answer encoders
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    question_encoder = Encoder(tokenizer.vocab_size, embed_size, 
                               output_embed_size)
    answer_encoder = Encoder(tokenizer.vocab_size, embed_size, 
                             output_embed_size)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, 
                                             shuffle=True)    
    optimizer = torch.optim.Adam(
        list(question_encoder.parameters()) + list(answer_encoder.parameters()
    ), lr=1e-5)
    loss_fn = torch.nn.CrossEntropyLoss()

    running_loss = []
    for _, data_batch in enumerate(dataloader):

        # Tokenize the question/answer pairs (each is a batch of 32 questions and 32 answers)
        question, answer = data_batch
        question_tok = tokenizer(question, padding=True, truncation=True, return_tensors='pt', max_length=max_seq_len)
        answer_tok = tokenizer(answer, padding=True, truncation=True, return_tensors='pt', max_length=max_seq_len)

        # Compute the embeddings: the output is of dim = 32 x 128
        question_embed = question_encoder(question_tok)
        answer_embed = answer_encoder(answer_tok)

        # Compute similarity scores: a 32x32 matrix
        # row[N] reflects similarity between question[N] and answers[0...31]
        similarity_scores = question_embed @ answer_embed.T

        # we want to maximize the values in the diagonal
        target = torch.arange(question_embed.shape[0], dtype=torch.long)
        loss = loss_fn(similarity_scores, target)
        running_loss += [loss.item()]

        # this is where the magic happens
        optimizer.zero_grad()    # reset optimizer so gradients are all-zero
        loss.backward()
        optimizer.step()

    return question_encoder, answer_encoder

In [32]:
def train(dataset, num_epochs=10):
    embed_size = 512
    output_embed_size = 128
    max_seq_len = 64
    batch_size = 32

    n_iters = len(dataset) // batch_size + 1
    
    # define the question/answer encoders
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    question_encoder = Encoder(tokenizer.vocab_size, embed_size, output_embed_size)
    answer_encoder = Encoder(tokenizer.vocab_size, embed_size, output_embed_size)

    # define the dataloader, optimizer and loss function    
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)    
    optimizer = torch.optim.Adam(list(question_encoder.parameters()) + list(answer_encoder.parameters()), lr=1e-5)
    loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        running_loss = []
        for idx, data_batch in enumerate(dataloader):

            # Tokenize the question/answer pairs (each is a batc of 32 questions and 32 answers)
            question, answer = data_batch
            question_tok = tokenizer(question, padding=True, truncation=True, return_tensors='pt', max_length=max_seq_len)
            answer_tok = tokenizer(answer, padding=True, truncation=True, return_tensors='pt', max_length=max_seq_len)
            if idx == 0 and epoch == 0:
                print(question_tok['input_ids'].shape, answer_tok['input_ids'].shape)
            
            # Compute the embeddings: the output is of dim = 32 x 128
            question_embed = question_encoder(question_tok)
            answer_embed = answer_encoder(answer_tok)
            if idx == 0 and epoch == 0:
                print(question_embed.shape, answer_embed.shape)
    
            # Compute similarity scores: a 32x32 matrix
            # row[N] reflects similarity between question[N] and answers[0...31]
            similarity_scores = question_embed @ answer_embed.T
            if idx == 0 and epoch == 0:
                print(similarity_scores.shape)
    
            # we want to maximize the values in the diagonal
            target = torch.arange(question_embed.shape[0], dtype=torch.long)
            loss = loss_fn(similarity_scores, target)
            running_loss += [loss.item()]
            if idx == n_iters-1:
                print(f"Epoch {epoch}, loss = ", np.mean(running_loss))
    
            # this is where the magic happens
            optimizer.zero_grad()    # reset optimizer so gradients are all-zero
            loss.backward()
            optimizer.step()

    return question_encoder, answer_encoder

In [28]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, datapath):
        self.data = pd.read_csv(datapath, sep="\t", nrows=300)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data.iloc[idx]['questions'], self.data.iloc[idx]['answers']

dataset = MyDataset('./nq_sample.tsv')
dataset.data.head(5)

,questions,answers
0,who played bubba in the tv series in the heat ...,Carlos Alan Autry Jr. (also known for a period...
1,where did the 2017 tour de france start,"The 3,540 km (2,200 mi)-long race commenced wi..."
2,who is the chess champion of the world,Current world champion Magnus Carlsen won the ...
3,who scored the most hat tricks in football,Cristiano Ronaldo and Messi have scored three ...
4,what do you need to be an ontario scholar,Ontario Scholars are high school graduates in ...


In [33]:
qe, ae = train(dataset, num_epochs=1000)

torch.Size([32, 20]) torch.Size([32, 64])
torch.Size([32, 128]) torch.Size([32, 128])
torch.Size([32, 32])
Epoch 0, loss =  3.8056437253952025
Epoch 1, loss =  3.6542222261428834
Epoch 2, loss =  3.4793503284454346
Epoch 3, loss =  3.41208930015564
Epoch 4, loss =  3.352181005477905
Epoch 5, loss =  3.3022516489028932
Epoch 6, loss =  3.247248911857605
Epoch 7, loss =  3.202755331993103
Epoch 8, loss =  3.1621811151504517
Epoch 9, loss =  3.1201109647750855
Epoch 10, loss =  3.045605754852295
Epoch 11, loss =  2.992043745517731
Epoch 12, loss =  2.931911313533783
Epoch 13, loss =  2.8537755846977233
Epoch 14, loss =  2.7627214431762694
Epoch 15, loss =  2.6462292075157166
Epoch 16, loss =  2.5960682153701784
Epoch 17, loss =  2.4110710501670836
Epoch 18, loss =  2.30485143661499
Epoch 19, loss =  2.1789428114891054
Epoch 20, loss =  2.038616418838501
Epoch 21, loss =  1.9149690508842467
Epoch 22, loss =  1.7356271266937255
Epoch 23, loss =  1.547472208738327
Epoch 24, loss =  1.3516941

In [34]:
question = 'What is the tallest mountain in the world?'
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
question_tok = tokenizer(question, padding=True, truncation=True, return_tensors='pt', max_length=64)
question_emb = qe(question_tok)[0]
print(question_tok)
print(question_emb[:5])


{'input_ids': tensor([[  101,  2054,  2003,  1996, 13747,  3137,  1999,  1996,  2088,  1029,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([ 0.5369, -0.9830, -0.4911,  0.0063,  0.4716], grad_fn=<SliceBackward0>)


In [35]:
answers = [
    "What is the tallest mountain in the world?",
    "The tallest mountain in the world is Mount Everest.",
    "Who is donald duck?"
]
answer_tok = []
answer_emb = []
for answer in answers:
    tok = tokenizer(answer, padding=True, truncation=True, return_tensors='pt', max_length=64)
    answer_tok.append(tok['input_ids'])
    emb = ae(tok)[0]
    answer_emb.append(emb)

print(answer_tok)
print(answer_emb[0][:5])
print(answer_emb[1][:5])
print(answer_emb[2][:5])

[tensor([[  101,  2054,  2003,  1996, 13747,  3137,  1999,  1996,  2088,  1029,
           102]]), tensor([[  101,  1996, 13747,  3137,  1999,  1996,  2088,  2003,  4057, 23914,
          1012,   102]]), tensor([[ 101, 2040, 2003, 6221, 9457, 1029,  102]])]
tensor([-0.2501, -0.2164, -0.4572, -1.0083, -1.0940], grad_fn=<SliceBackward0>)
tensor([-0.2816,  0.5321, -0.3513, -0.6424, -1.2049], grad_fn=<SliceBackward0>)
tensor([-0.2399,  1.0550, -0.5550, -0.5486, -0.6418], grad_fn=<SliceBackward0>)


In [36]:
question_emb @ torch.stack(answer_emb).T

tensor([19.0982, 19.1207,  5.6013], grad_fn=<SqueezeBackward4>)

In [38]:
question_emb

tensor([ 0.5369, -0.9830, -0.4911,  0.0063,  0.4716, -0.8067, -0.6319, -0.6080,
         0.8612, -0.7931,  0.2290, -0.4478,  0.0287, -0.3719, -0.6824, -0.3974,
         0.3811, -0.6123,  0.5428,  0.4425, -0.7947, -0.1094, -0.0270, -0.5699,
        -0.0487,  0.1897, -0.0379, -0.1851, -0.3665,  1.1892, -0.2309,  0.1031,
        -0.7128, -0.3312, -0.4424,  0.2279,  1.1627, -0.8449, -0.2117, -0.2811,
         0.4409, -0.3323,  1.4418,  0.4311,  1.1651,  0.9490,  0.8329,  0.8543,
        -0.4891,  0.2332,  1.3412,  0.5262,  0.3586, -0.1968,  0.2032,  0.7427,
        -0.6015,  0.4691, -1.9989,  0.2508, -0.7109,  0.4226, -0.3274,  1.2037,
         1.0752, -0.8101,  0.2781, -1.3233, -0.1118,  0.2100, -0.0293,  0.5646,
         1.4073, -0.5325,  0.0978, -1.0558, -0.3634, -0.6029, -0.2275,  0.0205,
         0.7554,  1.2805,  0.7450, -0.0819, -0.5603,  0.1785,  0.9935, -0.5530,
        -0.4441, -0.3574,  0.1243, -1.2339,  1.4053,  0.0412, -0.0520, -1.0860,
        -0.5161,  0.3242, -0.1244, -0.64

In [41]:
answer_emb[0]

tensor([-0.2501, -0.2164, -0.4572, -1.0083, -1.0940, -0.2121,  0.8114, -0.0475,
         1.1148,  0.1397,  0.0955, -0.1027, -0.2522,  0.0049,  0.7735, -1.4735,
        -0.3905, -1.1025,  0.0728, -0.1905, -0.7496, -0.2794, -0.2945,  0.0621,
        -0.2239,  0.1714,  0.4438,  0.2449,  0.8586, -0.1381,  0.5936, -0.1552,
         0.6306,  0.7641, -0.8883,  0.4047,  0.8318,  0.5008,  0.8376,  0.7296,
        -0.0227,  0.1188,  1.8296, -0.2360, -0.1740,  1.3084,  1.4496,  0.5564,
        -0.3819, -0.2451,  1.1446,  1.0633, -0.4572, -0.4464, -0.1048,  0.0506,
        -0.4418,  0.3070, -1.0491,  1.0227,  0.1769,  0.5745, -0.2467,  1.0437,
         0.3254, -0.6818, -0.6153, -0.2549,  0.0243,  0.1709,  0.3619, -0.0190,
         0.3557,  0.2329, -0.4337, -1.0137, -0.7572, -0.0143, -0.8741,  1.0119,
         0.2730,  0.4572, -0.3193,  0.6547, -0.7241, -0.0554,  0.4251, -0.5491,
        -0.1839,  0.1560, -0.2798, -0.5794, -1.0352, -1.3029, -0.9109, -1.1015,
        -1.5596, -0.0171,  0.9850, -0.91

## RAG application

The motivation during RAG is you are looking for the best Answer for the query not just the most similar. For example, if you use same embedding for both question and answer, you will get the best result as itself as shown using just sentence embedding for both question and answer. 

But if you use answer embedding for all answers and embed the question with another embedding model, then you will get the retrieval result with the best answer not the question itself as shown below.

In [6]:
answers = [
    "What are the seven deadly sins",
    "The seven deadly sins—pride, greed, lust, envy, gluttony, wrath, and sloth",
    "Louis-Napoléon Bonaparte (also known as Louis Napoleon or Napoleon III) was the first president of France."
]
question =  "What are the seven deadly sins"
    

In [8]:
sbert_ans_embeds = [sbert_model.encode(answer) for answer in answers]
sbert_q_embeds = sbert_model.encode(question)

In [11]:
for i, ans in enumerate(answers):
    print(f"Similarity between {ans} and {question}: {cosine_similarity([sbert_ans_embeds[i]],[sbert_q_embeds])[0][0]}\n ")
    

Similarity between What are the seven deadly sins and What are the seven deadly sins: 1.0
 
Similarity between The seven deadly sins—pride, greed, lust, envy, gluttony, wrath, and sloth and What are the seven deadly sins: 0.8332808017730713
 
Similarity between Louis-Napoléon Bonaparte (also known as Louis Napoleon or Napoleon III) was the first president of France. and What are the seven deadly sins: 0.06011902913451195
 


In [13]:
answer_tokenizer = AutoTokenizer \
                   .from_pretrained("facebook/dpr-ctx_encoder-multiset-base")
answer_encoder = DPRContextEncoder \
                   .from_pretrained("facebook/dpr-ctx_encoder-multiset-base")

question_tokenizer = AutoTokenizer \
                   .from_pretrained("facebook/dpr-question_encoder-multiset-base")
question_encoder = DPRQuestionEncoder \
                   .from_pretrained("facebook/dpr-question_encoder-multiset-base")


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-multiset-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [16]:
question_embed = question_encoder(question_tokenizer(question, return_tensors="pt")["input_ids"]).pooler_output.flatten().tolist()
answer_embeds = [
    answer_encoder(answer_tokenizer(ans, return_tensors="pt")["input_ids"]).pooler_output.flatten().tolist()
    for ans in answers
]

In [18]:
for i, ans in enumerate(answers):
    print(f"Similarity between {ans} and {question}: {cosine_similarity([answer_embeds[i]],[question_embed])[0][0]}\n ")
    

Similarity between What are the seven deadly sins and What are the seven deadly sins: 0.7224752240350718
 
Similarity between The seven deadly sins—pride, greed, lust, envy, gluttony, wrath, and sloth and What are the seven deadly sins: 0.7543505209494019
 
Similarity between Louis-Napoléon Bonaparte (also known as Louis Napoleon or Napoleon III) was the first president of France. and What are the seven deadly sins: 0.3482125850905844
 
